In [3]:
import shioaji as sj
import pandas as pd
import numpy as np
import datetime
import time
from shioaji import TickFOPv1, BidAskFOPv1, Exchange
from datetime import timedelta

2024-05-27 20:29:31.562 | WARNING  | importlib._bootstrap:_call_with_frames_removed:219 - Optional: pip install shioaji[speed] for better performance.


In [9]:
api = sj.Shioaji(simulation=True)
api.login(
    api_key="J1txJtRBFJx3uCDwMG1DE47oe2QJeEZVSAaTq1uvSxCB", 
    secret_key="GSWPQadvP4TWm7fpbDjy7F5Jh7Wv7JGSjABNYaqxUn7Q"
)
api.usage()

Response Code: 0 | Event Code: 0 | Info: host '203.66.91.161:80', hostname '203.66.91.161:80' IP 203.66.91.161:80 (host 1 of 1) (host connection attempt 1 of 1) (total connection attempt 1 of 1) | Event: Session up


UsageStatus(connections=2, bytes=18028924, limit_bytes=524288000, remaining_bytes=506259076)

In [10]:
api.Contracts.Futures.TXF.TXFR1

Future(code='TXFR1', symbol='TXFR1', name='臺股期貨近月', category='TXF', delivery_month='202404', delivery_date='2024/04/17', underlying_kind='I', unit=1, limit_up=22222.0, limit_down=18182.0, reference=20202.0, update_date='2024/03/29', target_code='TXFD4')

1.2024/03/19台指期Tick行情數據，並轉成DataFrame

In [11]:
ticks = api.ticks(
    contract=api.Contracts.Futures.TXF.TXFR1, 
    date="2024-03-19"
)
#print(ticks)
df = pd.DataFrame({**ticks})
df.ts = pd.to_datetime(df.ts)
df.set_index('ts',inplace=True)
df

,close,volume,bid_price,bid_volume,ask_price,ask_volume,tick_type
ts,,,,,,,
2024-03-18 15:00:00.009,19917.0,61,19914.0,24,19919.0,4,1
2024-03-18 15:00:00.011,19916.0,3,19916.0,3,19917.0,54,1
2024-03-18 15:00:00.019,19917.0,3,19916.0,3,19917.0,54,1
2024-03-18 15:00:00.019,19917.0,3,19916.0,3,19917.0,54,1
2024-03-18 15:00:00.019,19917.0,1,19916.0,3,19917.0,54,1
...,...,...,...,...,...,...,...
2024-03-19 13:44:59.575,19867.0,1,19867.0,8,19870.0,8,2
2024-03-19 13:44:59.719,19870.0,3,19867.0,7,19870.0,8,1
2024-03-19 13:44:59.727,19870.0,1,19867.0,7,19870.0,8,1


2.2023/01/01 ~ 2024/03/18台指期Kbars行情數據，並轉成DataFrame

In [13]:
kbars= api.kbars(
    contract=api.Contracts.Futures.TXF.TXFR1, 
    start="2023-01-01", 
    end="2024-03-18", 
)
df =pd.DataFrame({**kbars})
df.ts= pd.to_datetime(df.ts)
df.set_index('ts',inplace=True)
df

,Open,High,Low,Close,Volume,Amount
ts,,,,,,
2023-01-03 08:46:00,14050.0,14063.0,14043.0,14048.0,2230,31337041.0
2023-01-03 08:47:00,14048.0,14051.0,14039.0,14050.0,905,12709817.0
2023-01-03 08:48:00,14050.0,14052.0,14046.0,14050.0,490,6883821.0
2023-01-03 08:49:00,14050.0,14051.0,14038.0,14039.0,427,5996528.0
2023-01-03 08:50:00,14038.0,14048.0,14033.0,14042.0,597,8381717.0
...,...,...,...,...,...,...
2024-03-18 23:55:00,19864.0,19869.0,19863.0,19869.0,115,2284533.0
2024-03-18 23:56:00,19867.0,19872.0,19865.0,19872.0,87,1728530.0
2024-03-18 23:57:00,19872.0,19876.0,19872.0,19874.0,129,2563734.0


3.將2.之Kbars數據取開盤價(Open)、最高價(High)、最低價(Low)、收盤價(Close)、成交量(Volume)，並轉成5分K

In [14]:
def kbars_set_interval(kbars_df,interval='5Min'):
    return_df=pd.DataFrame()
    kbars_df.index = kbars_df.index + timedelta(minutes = -1)   
    if  interval=='1Min':
        return kbars_df
    else:    
        return_df['Open'] = kbars_df['Open'].resample(interval).first()
        return_df['High'] = kbars_df['High'].resample(interval).max()
        return_df['Low'] = kbars_df['Low'].resample(interval).min()
        return_df['Close'] = kbars_df['Close'].resample(interval).last()
        return_df['Volume'] = kbars_df['Volume'].resample(interval).sum()
        return_df.dropna(inplace=True)
        return return_df

In [15]:
kbars_set_interval(df,'5Min')

,Open,High,Low,Close,Volume
ts,,,,,
2023-01-03 08:45:00,14050.0,14063.0,14033.0,14042.0,4649
2023-01-03 08:50:00,14041.0,14054.0,14037.0,14042.0,1197
2023-01-03 08:55:00,14042.0,14043.0,14027.0,14031.0,1857
2023-01-03 09:00:00,14032.0,14041.0,14013.0,14027.0,3327
2023-01-03 09:05:00,14027.0,14039.0,14008.0,14021.0,2886
...,...,...,...,...,...
2024-03-18 23:35:00,19876.0,19890.0,19874.0,19889.0,498
2024-03-18 23:40:00,19889.0,19892.0,19885.0,19889.0,266
2024-03-18 23:45:00,19889.0,19893.0,19881.0,19882.0,343


4.即時台指期開盤價(Open)、最高價(High)、最低價(Low)、收盤價(Close)、成交量(Volume)的數據

In [16]:
@api.on_tick_fop_v1()
def quote_callback(exchange:Exchange, tick:TickFOPv1):
    print(f"Exchange: {exchange}, Open: {tick.open}, High: {tick.high}, Low: {tick.low}, Open: {tick.open}, Volume: {tick.volume}")

In [17]:
api.quote.subscribe(
    api.Contracts.Futures.TXF["TXFR1"],
    quote_type = sj.constant.QuoteType.Tick,
    version = sj.constant.QuoteVersion.v1,
)

Response Code: 200 | Event Code: 16 | Info: TIC/v1/FOP/*/TFE/TXFD4 | Event: Subscribe or Unsubscribe ok
Exchange: TAIFEX, Open: 20226, High: 20299, Low: 20192, Open: 20226, Volume: 6
Exchange: TAIFEX, Open: 20226, High: 20299, Low: 20192, Open: 20226, Volume: 1
Exchange: TAIFEX, Open: 20226, High: 20299, Low: 20192, Open: 20226, Volume: 1
Exchange: TAIFEX, Open: 20226, High: 20299, Low: 20192, Open: 20226, Volume: 1
Exchange: TAIFEX, Open: 20226, High: 20299, Low: 20192, Open: 20226, Volume: 1
Exchange: TAIFEX, Open: 20226, High: 20299, Low: 20192, Open: 20226, Volume: 1


In [18]:
#取消訂閱
api.quote.unsubscribe(api.Contracts.Futures.TXF["TXFR1"], quote_type='tick')

Response Code: 200 | Event Code: 16 | Info: TIC/v1/FOP/*/TFE/TXFD4 | Event: Subscribe or Unsubscribe ok


5.即時台指期掛單上下五檔價量的數據

In [19]:
@api.on_bidask_fop_v1()
def quote_callback(exchange:Exchange, bidask:BidAskFOPv1):
    print(f"Exchange: {exchange}, Bid Price: {bidask.bid_price}, Bid Volume: {bidask.bid_volume}, Ask Price: {bidask.ask_price}, Ask Volume: {bidask.ask_volume}")

In [20]:
api.quote.subscribe(
    api.Contracts.Futures.TXF.TXFR1,
    quote_type = sj.constant.QuoteType.BidAsk,
    version = sj.constant.QuoteVersion.v1,
)

Response Code: 200 | Event Code: 16 | Info: QUO/v1/FOP/*/TFE/TXFD4 | Event: Subscribe or Unsubscribe ok
Exchange: TAIFEX, Bid Price: [Decimal('20277'), Decimal('20276'), Decimal('20275'), Decimal('20274'), Decimal('20273')], Bid Volume: [5, 16, 28, 24, 34], Ask Price: [Decimal('20278'), Decimal('20279'), Decimal('20280'), Decimal('20281'), Decimal('20282')], Ask Volume: [2, 43, 33, 26, 43]
Exchange: TAIFEX, Bid Price: [Decimal('20277'), Decimal('20276'), Decimal('20275'), Decimal('20274'), Decimal('20273')], Bid Volume: [5, 16, 28, 24, 34], Ask Price: [Decimal('20278'), Decimal('20279'), Decimal('20280'), Decimal('20281'), Decimal('20282')], Ask Volume: [2, 43, 33, 26, 43]
Exchange: TAIFEX, Bid Price: [Decimal('20277'), Decimal('20276'), Decimal('20275'), Decimal('20274'), Decimal('20273')], Bid Volume: [5, 16, 30, 24, 34], Ask Price: [Decimal('20278'), Decimal('20279'), Decimal('20280'), Decimal('20281'), Decimal('20282')], Ask Volume: [1, 43, 33, 26, 43]


In [21]:
#取消訂閱
api.quote.unsubscribe(api.Contracts.Futures.TXF["TXFR1"], quote_type='bidask')

Exchange: TAIFEX, Bid Price: [Decimal('20277'), Decimal('20276'), Decimal('20275'), Decimal('20274'), Decimal('20273')], Bid Volume: [5, 16, 30, 24, 34], Ask Price: [Decimal('20278'), Decimal('20279'), Decimal('20280'), Decimal('20281'), Decimal('20282')], Ask Volume: [1, 43, 33, 26, 43]Response Code: 200 | Event Code: 16 | Info: QUO/v1/FOP/*/TFE/TXFD4 | Event: Subscribe or Unsubscribe ok



In [22]:
api.logout()

True